In [ ]:
# Install Tiny-SIMA and the optional VLM backend into this kernel.
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Leave room for the PyTorch VLM before importing JAX.
os.environ.setdefault(
    "XLA_PYTHON_CLIENT_MEM_FRACTION",
    os.environ.get("TINY_SIMA_JAX_MEMORY_FRACTION", "0.35"),
)

if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError("Tiny-SIMA requires Python 3.11 or 3.12.")

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the Tiny-SIMA repository.")

cuda_override = os.environ.get("TINY_SIMA_USE_CUDA")
if cuda_override is None:
    has_cuda = (
        shutil.which("nvidia-smi") is not None
        and subprocess.run(
            ["nvidia-smi"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=False,
        ).returncode == 0
    )
else:
    has_cuda = cuda_override.lower() in {"1", "true", "yes"}

extras = "notebook,vlm,cuda12" if has_cuda else "notebook,vlm"
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--no-cache-dir",
    "--disable-pip-version-check",
    "-e",
    f"{repo_root}[{extras}]",
])
# Reload edited Tiny-SIMA source on every cell execution, so iterating on the
# planner prompt costs one cell run instead of a kernel restart -- which here
# means reinstalling the package and reloading the VLM weights. Existing
# objects keep working because autoreload patches methods onto the live
# classes, and a prompt edit changes the response cache key, so the scorer
# cache cannot serve an answer to the old prompt.
ipython = get_ipython()
ipython.run_line_magic("load_ext", "autoreload")
ipython.run_line_magic("autoreload", "2")

# Import Torchvision before Transformers. Transformers caches optional
# dependency availability on first import.
torch = importlib.import_module("torch")
torchvision = importlib.import_module("torchvision")
transformers = importlib.import_module("transformers")
outlines = importlib.import_module("outlines")
transformers_utils = importlib.import_module("transformers.utils")

print(
    "PyTorch:", torch.__version__,
    "Torchvision:", torchvision.__version__,
    "Transformers:", transformers.__version__,
    "Outlines: installed",
)
if not transformers_utils.is_torchvision_available():
    raise RuntimeError(
        "Transformers cached Torchvision as unavailable. Restart this "
        "kernel, then run this installation cell before any other cell."
    )

source_root = str(repo_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

print("Repository:", repo_root)
print("CUDA detected:", has_cuda)
print("JAX memory fraction:", os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"])

# Phase 4: hierarchical VLM planner

This notebook evaluates a frozen VLM at skill boundaries. The model supplies the Say term (objective usefulness); the Can term comes from horizon-bounded reachability and a measured per-skill competence prior. The planner never sees the private scripted reference plan. Start it in a fresh kernel so the JAX memory fraction is applied before JAX initializes.

The evaluation has four layers:

1. Skill probing labels every skill at every chain boundary with whether the executor actually finished it, on device and batched.
2. Two competence heads are fitted to those labels and scored on a **different** seed: the critic-as-affordance recalibration, which this measurement retired, and the `(skill, reachable, steps_to_target)` prior that replaced it.
3. Controlled-prefix evaluation isolates planner decisions by letting the scripted executor advance the world after each prediction.
4. Closed-loop evaluation connects the planner to the learned stochastic executor and measures objective success.

Two negative results shape the ladder below, and both are reproduced here rather than asserted.

Symbolic preconditions cannot supply a useful Can term: they test whether a block exists anywhere on the 64x64 map, which is true in essentially every state, so the planner ranked unreachable skills above reachable ones.

Neither can the executor's critic. Measured on 432 labelled probes it separates success at AUC 0.43 - worse than chance - because the skill curriculum always places a reachable target, so the value function encodes which skill was requested rather than whether this state admits it. The isotonic fit therefore collapses onto the base rate and multiplies every candidate by roughly the same number, which cannot reorder candidates whose rank utilities differ fourfold. The same labels do carry signal in reachability (success 0.70 against 0.03) and in skill identity (0.17 to 0.63), which is what the competence prior reads.

That leaves the VLM itself as the dominant error term, so the ladder also measures the prompt: `vlm_baseline_prompt` is the exact prompt that produced a 74.5% ungrounded top choice, and the default `available_first` style makes the model report what is available before the constrained decoder lets it name a choice.

In [ ]:
import json
import time

import jax
import matplotlib.pyplot as plt

from tiny_sima.config import load_vlm_planner_config
from tiny_sima.envs.chains import make_skill_chain_environment
from tiny_sima.envs.craftax import render_state
from tiny_sima.evaluation import (
    WOOD_TO_STONE_CHAIN,
    collect_affordance_samples,
    collect_prefix_trajectory,
    evaluate_affordance_calibration,
    evaluate_competence_prior,
    evaluate_planner_prefixes,
    evaluate_skill_chain,
    fit_affordance_calibration,
    fit_affordance_competence_prior,
    load_goal_conditioned_checkpoint,
    make_learned_skill_executor,
    run_vlm_smoke_test,
)
from tiny_sima.goals import (
    DEFAULT_REACHABILITY,
    SKILL_IDS,
    skill_reachability_report,
)
from tiny_sima.planner import (
    CompetencePrior,
    IsotonicCalibration,
    PlannerContext,
    PlannerPromptStyle,
    SayCanPlanner,
    SequentialUtilityScorer,
    TransformersVLMBackend,
    VLMUtilityScorer,
    build_planner_prompt,
    build_planner_system_prompt,
    constant_feasibility,
    constant_utility,
    make_critic_affordance_scorer,
    reachability_feasibility,
    symbolic_feasibility,
)

print("JAX devices:", jax.devices())
print("Reachability view radius:", DEFAULT_REACHABILITY.view_radius)
print("Unknown-target feasibility:", DEFAULT_REACHABILITY.unknown_feasibility)
print("Default prompt style:", PlannerPromptStyle.AVAILABLE_FIRST.value)

In [ ]:
CHECKPOINT = repo_root / "runs" / "skill-ppo-local-1m" / "checkpoints" / "final"
VLM_CONFIG_PATH = repo_root / "configs" / "vlm_planner_qwen3_2b.yaml"
RUN_DIR = repo_root / "runs" / "skill-ppo-local-1m"
CALIBRATION_PATH = RUN_DIR / "affordance_calibration.json"
COMPETENCE_PRIOR_PATH = RUN_DIR / "competence_prior.json"

# Iteration preset. "prompt" is the loop to run while editing the prompt or the
# response schema: full prefix episodes, no probing (both fits load from disk),
# no closed loop. "smoke" is the cheapest end-to-end pass over every cell, and
# "full" is the reported run. Any knob below can still be overridden by hand.
EVAL_PRESET = "prompt"

EVAL_PRESETS = {
    "smoke": {
        "prefix_episodes": 1,
        "closed_loop_episodes": 2,
        "calibration_episodes": 2,
        "run_probing": True,
        "run_closed_loop": True,
    },
    "prompt": {
        "prefix_episodes": 8,
        "closed_loop_episodes": 0,
        "calibration_episodes": 0,
        "run_probing": False,
        "run_closed_loop": False,
    },
    "full": {
        "prefix_episodes": 8,
        # Closed-loop episodes run in lockstep, so the count costs batch width
        # rather than wall clock; 64 gives a standard error near 0.05.
        "closed_loop_episodes": 64,
        "calibration_episodes": 12,
        "run_probing": True,
        "run_closed_loop": True,
    },
}
if EVAL_PRESET not in EVAL_PRESETS:
    raise ValueError(f"EVAL_PRESET must be one of {sorted(EVAL_PRESETS)}.")
preset = EVAL_PRESETS[EVAL_PRESET]

RUN_VLM_SMOKE_TEST = True
# Probing is a measurement of this checkpoint, not of the prompt, so it runs
# once and both fits are reloaded from disk afterwards.
RUN_AFFORDANCE_CALIBRATION = preset["run_probing"]
RUN_CLOSED_LOOP = preset["run_closed_loop"]
# Prefix episodes decode in lockstep: one batched generation per boundary
# covers every surviving episode, so episodes cost memory rather than latency.
NUM_PREFIX_EPISODES = preset["prefix_episodes"]
NUM_CLOSED_LOOP_EPISODES = preset["closed_loop_episodes"]
# Each episode probes all six skills at every chain boundary. The probes run
# batched on device, so the cost is dominated by the scripted advance between
# boundaries rather than by episodes x boundaries x skills.
NUM_CALIBRATION_EPISODES = preset["calibration_episodes"]
# Both heads are fitted on CALIBRATION_SEED and scored on CALIBRATION_EVAL_SEED.
# Scoring a head on the samples it was fitted on measures separability, not
# generalization, and makes the reported calibration error meaningless.
CALIBRATION_SEED = 7
CALIBRATION_EVAL_SEED = 23
SKILL_HORIZON = 64
EVALUATION_SEED = 51

vlm_config = load_vlm_planner_config(VLM_CONFIG_PATH)
print("VLM:", vlm_config.model_id)
print("Checkpoint exists:", CHECKPOINT.exists())
print("Preset:", EVAL_PRESET, preset)
print("VLM batch size:", vlm_config.max_batch_size)

## Inspect the planner input

The VLM receives a rendered frame plus a compact text record. This cell is cheap and should be inspected before loading model weights.

In [ ]:
preview_environment = make_skill_chain_environment()
preview_state = preview_environment.reset(jax.random.PRNGKey(EVALUATION_SEED)).state
preview_pixels = render_state(
    preview_state,
    vlm_config.environment_name,
    block_pixel_size=vlm_config.block_pixel_size,
)
plt.figure(figsize=(8, 6))
plt.imshow(preview_pixels)
plt.axis("off")
plt.show()

# Both prompts are printed because the ablation measures the difference between
# them. BASELINE is the prompt that produced ungrounded_top = 0.745 and is kept
# byte for byte; AVAILABLE_FIRST hands the model the availability it was being
# asked to infer and makes it echo that list before the schema lets it choose.
for style in PlannerPromptStyle:
    preview_prompt = build_planner_prompt(
        preview_state,
        context=PlannerContext(
            objective=WOOD_TO_STONE_CHAIN.objective,
        ),
        candidate_skills=SKILL_IDS,
        style=style,
    )
    print("=" * 78)
    print(f"STYLE: {style.value}")
    print("\nSYSTEM:\n", build_planner_system_prompt(style))
    print("\nUSER:\n", preview_prompt)

In [ ]:
# Cheap grounding check: compare what the precondition knows against what is
# actually within reach. The precondition is true whenever a block exists
# anywhere on the 64x64 map, which is why it ranked unreachable skills first.
preview_context = PlannerContext(objective=WOOD_TO_STONE_CHAIN.objective)
preview_steps = skill_reachability_report(preview_state, DEFAULT_REACHABILITY)
preview_graded = reachability_feasibility.score_all_feasibility(
    preview_state,
    preview_context,
    SKILL_IDS,
)
print(f"{'skill':22s} {'precondition':>12s} {'steps':>7s} {'graded':>8s}")
for skill_id in SKILL_IDS:
    steps = preview_steps[skill_id]
    print(
        f"{skill_id.value:22s} "
        f"{symbolic_feasibility(preview_state, preview_context, skill_id):12.2f} "
        f"{'-' if steps is None else steps:>7} "
        f"{preview_graded[skill_id]:8.2f}"
    )

## End-to-end VLM smoke test

This performs one real multimodal generation and verifies constrained decoding, complete-ranking validation, and symbolic grounding before the longer evaluations run.

In [ ]:
backend = TransformersVLMBackend(vlm_config)
vlm_utility = VLMUtilityScorer(backend=backend, config=vlm_config)
# A separate scorer, because the response cache is per scorer and the two
# prompt styles must not share generations. One backend, so the model weights
# are loaded once; the constrained generator is cached per schema.
vlm_baseline_utility = VLMUtilityScorer(
    backend=backend,
    config=vlm_config,
    prompt_style=PlannerPromptStyle.BASELINE,
)
# The position-bias control: the same prompt with the available skills listed
# in a seeded order rather than catalog order, which always puts collect_wood
# first. Its own scorer, because the prompts and so the generations differ.
vlm_shuffled_utility = VLMUtilityScorer(
    backend=backend,
    config=vlm_config,
    available_shuffle_seed=EVALUATION_SEED,
)

if RUN_VLM_SMOKE_TEST:
    smoke_result = run_vlm_smoke_test(
        vlm_config,
        scorer=vlm_utility,
        seed=EVALUATION_SEED,
    )
    print(smoke_result.to_dict())

## Probe the skills, then fit two competence heads

The skill environment pays one sparse reward on success and terminates there, so the goal-conditioned critic estimates `E[gamma ** k] * P(success)`: in principle monotone in success probability, squashed by discounting. That was the hypothesis. The cell below tests it instead of assuming it, by probing every skill at every boundary from a copy of the boundary state, recording what the critic predicted, and fitting an isotonic map from critic value to success.

The measurement refuted it. Watch `auc` in the per-skill table: pooled it came out at 0.43 and per skill at roughly 0.5. The reason is in the training distribution - `local_navigation` always spawns a reachable target, so the critic never saw a state where its skill was impossible and learned to encode skill identity instead. An isotonic fit must be non-decreasing, but the cross-skill relation between value and success is decreasing, so the fit flattens to the base rate and the Can term becomes a near-constant multiplier. Rank utilities differ by 4x between adjacent positions; a factor that varies by 1.15x cannot reorder anything.

So the same labels feed a second head. `fit_affordance_competence_prior` tabulates success by `(skill, reachable, distance band)` - the three features the labels show carry signal - and shrinks each cell toward its parent so a thin cell inherits an estimate instead of reading 0 or 1. Samples whose symbolic precondition failed are dropped, because the precondition vetoes at scoring time anyway.

Both heads are fitted on `CALIBRATION_SEED` and scored on `CALIBRATION_EVAL_SEED`. The in-sample ECE of an isotonic fit is identically zero by construction, so a single-seed protocol reports a perfect number for a head that carries no information.

`unreachable_successes` is the other number worth watching: it counts skills the reachability predicate called unreachable that the executor completed anyway by walking out of the observation window. Every one is an episode a hard gate would have thrown away, which is why both the reachability term and the competence prior grade rather than veto.

The fits are saved so a later session can skip re-probing: `critic_affordance_scorer_from_checkpoint(CHECKPOINT, calibration_path=CALIBRATION_PATH)` for the retired ablation, `CompetencePrior.load(COMPETENCE_PRIOR_PATH)` for the head in use.

In [ ]:
if not CHECKPOINT.exists():
    raise FileNotFoundError(f"Missing executor checkpoint: {CHECKPOINT}")

# One load shared by the affordance scorer and the closed-loop executor, so the
# critic being measured belongs to the policy under test.
checkpoint = load_goal_conditioned_checkpoint(CHECKPOINT)
executor = make_learned_skill_executor(checkpoint, deterministic=False)
print("gamma:", checkpoint.config.gamma)
print("training skill_horizon:", checkpoint.config.skill_horizon)

if RUN_AFFORDANCE_CALIBRATION:
    uncalibrated_affordance = make_critic_affordance_scorer(checkpoint)
    started = time.perf_counter()
    fit_samples = collect_affordance_samples(
        uncalibrated_affordance,
        executor,
        num_episodes=NUM_CALIBRATION_EPISODES,
        seed=CALIBRATION_SEED,
        skill_horizon=SKILL_HORIZON,
    )
    held_out_samples = collect_affordance_samples(
        uncalibrated_affordance,
        executor,
        num_episodes=NUM_CALIBRATION_EPISODES,
        seed=CALIBRATION_EVAL_SEED,
        skill_horizon=SKILL_HORIZON,
    )
    probe_seconds = time.perf_counter() - started

    calibration = fit_affordance_calibration(fit_samples)
    calibration.save(CALIBRATION_PATH)
    competence_prior = fit_affordance_competence_prior(fit_samples)
    competence_prior.save(COMPETENCE_PRIOR_PATH)

    report = evaluate_affordance_calibration(held_out_samples, calibration)
    competence_report = evaluate_competence_prior(held_out_samples, competence_prior)
    print(
        f"fit_samples={len(fit_samples)} "
        f"held_out_samples={report['num_samples']} "
        f"held_out={report['held_out']} "
        f"seconds={probe_seconds:.1f}"
    )
    print("critic overall:", json.dumps(report["overall"], indent=2, default=float))
    print(
        "competence overall:",
        json.dumps(competence_report["overall"], indent=2, default=float),
    )
    print(
        f"base_rate={competence_report['base_rate']:.3f} "
        f"unreachable_rate={report['unreachable_rate']:.3f} "
        f"unreachable_successes={report['unreachable_successes']} "
        f"reachable_success_rate={report['reachable_success_rate']:.3f} "
        f"unreachable_success_rate={report['unreachable_success_rate']:.3f}"
    )
    # Two populations, each read against its own success rate. The critic is
    # scored on every probe; the prior only on probes whose precondition held,
    # because the precondition vetoes the rest at scoring time. Reading the
    # prior against the all-probe rate made a well-calibrated place_table
    # look like 0.98 predicted against 0.17 observed.
    print(
        f"\n{'':22s} {'critic, all probes':>30s} | "
        f"{'prior, precondition held':>24s}"
    )
    print(
        f"{'skill':22s} {'n':>4s} {'mean_value':>10s} {'auc':>6s} "
        f"{'success':>7s} | {'n':>4s} {'prior_p':>8s} {'success':>8s}"
    )
    for skill_name, metrics in report["per_skill"].items():
        prior_metrics = competence_report["per_skill"].get(skill_name, {})
        print(
            f"{skill_name:22s} {int(metrics['num_samples']):4d} "
            f"{metrics['mean_critic_value']:10.3f} {metrics['auc']:6.3f} "
            f"{metrics['success_rate']:7.3f} | "
            f"{int(prior_metrics.get('num_samples', 0)):4d} "
            f"{prior_metrics.get('mean_probability', float('nan')):8.3f} "
            f"{prior_metrics.get('success_rate', float('nan')):8.3f}"
        )
else:
    calibration = (
        IsotonicCalibration.load(CALIBRATION_PATH)
        if CALIBRATION_PATH.exists()
        else None
    )
    competence_prior = (
        CompetencePrior.load(COMPETENCE_PRIOR_PATH)
        if COMPETENCE_PRIOR_PATH.exists()
        else None
    )
    competence_report = None

calibrated_affordance = make_critic_affordance_scorer(
    checkpoint,
    calibration=calibration,
)

In [ ]:
# The fitted table, cell by cell. `fit_support` is how many probes the cell was
# estimated from and `probability` is the shrunk estimate, so a small support
# next to a probability near its neighbours is shrinkage doing its job rather
# than evidence. `success_rate` is the held-out outcome on the same rows.
if competence_prior is None:
    print("No competence prior available; run the cell above or fit one.")
else:
    print(
        f"{'skill':22s} {'band':>18s} {'fit_n':>6s} {'p':>6s} "
        f"{'eval_n':>7s} {'success':>8s}"
    )
    rows = competence_report["cells"] if competence_report is not None else []
    for cell in rows:
        print(
            f"{cell['skill']:22s} {cell['label']:>18s} "
            f"{cell['fit_support']:6d} {cell['probability']:6.3f} "
            f"{cell['num_samples']:7d} {cell['success_rate']:8.3f}"
        )
    if not rows:
        print("(loaded from disk; re-run probing to score it on held-out states)")
    print("\ndistance bands:", competence_prior.distance_edges)
    print("floor:", competence_prior.floor)
    print("fitted base rate:", round(competence_prior.base_rate, 3))

## Controlled-prefix planner ablation

The same randomized states are used for every planner, and because the scripted oracle ignores the planner they are collected once, as a `PrefixTrajectory`, and every condition is scored on it. Episodes advance in lockstep, so one batched generation covers every episode's decision at a boundary, and the response cache means conditions sharing a scorer share generations. The ladder therefore costs roughly one batched pass per boundary per distinct prompt - three prompts, three passes - rather than one generation per episode per boundary.

The conditions isolate where grounding comes from:

| condition | Say | Can |
| --- | --- | --- |
| `scripted_oracle` | private reference sequence | reachability |
| `vlm_only` | VLM | none |
| `feasibility_only` | uniform | reachability |
| `vlm_plus_symbolic` | VLM | map-wide preconditions |
| `vlm_plus_reachability` | VLM | preconditions and reachability |
| `vlm_plus_competence_prior` | VLM | measured `(skill, reachable, steps)` success rates |
| `vlm_plus_critic_affordance` | VLM | reachability times calibrated critic (retired) |
| `vlm_baseline_prompt` | VLM, baseline prompt | preconditions and reachability |
| `vlm_shuffled_available` | VLM, available skills in seeded order | preconditions and reachability |

`vlm_plus_critic_affordance` is kept only as the ablation that measured its own failure. Its score should sit on top of `vlm_plus_reachability` rather than above it; if it does, that is the near-constant multiplier being visible end to end.

`vlm_baseline_prompt` differs from `vlm_plus_reachability` in the prompt alone, so the gap between their `ungrounded_top` values is the prompt effect with the Can term held fixed.

`vlm_shuffled_available` differs from `vlm_plus_reachability` only in the order the available skills are listed. In the first full run the model chose collect_wood over an available collect_stone at six of eight final boundaries, and catalog order always lists collect_wood first. If that is position bias, shuffling moves the error off the last boundary; if accuracy there is unchanged, the model is misreading progress and the fix belongs in the prompt's reasoning, not its order. The per-boundary table below the ladder is where to read it.

Rankings that repeat a skill are repaired at parse time - first occurrences kept, omitted skills appended - because the schema fixes the ranking's length but not that its entries are distinct, and the available-first model filled the slots by cycling the available skills (10 of 48 boundaries, every `selected_skill` correct). `repaired` is the share of boundaries that needed it and `strict` is the accuracy strict parsing would have given, read from the same generations.

Three metrics matter beyond accuracy. `ungrounded_top` is the share of boundaries where a reachable skill existed and the VLM still ranked an unreachable one first - the dominant error, and it is identical across conditions that share a utility scorer because it measures the VLM alone. `rescued` is how often the Can term then recovered an accepted skill. `grounded` is how often the chosen skill was usable from that exact state, which the old precondition-based definition could not measure because it was satisfied almost everywhere.

In [ ]:
prefix_planners = {
    "scripted_oracle": SayCanPlanner(
        utility_scorer=SequentialUtilityScorer(WOOD_TO_STONE_CHAIN.reference_skills),
    ),
    "vlm_only": SayCanPlanner(
        utility_scorer=vlm_utility,
        feasibility_scorer=constant_feasibility,
    ),
    "feasibility_only": SayCanPlanner(utility_scorer=constant_utility),
    "vlm_plus_symbolic": SayCanPlanner(
        utility_scorer=vlm_utility,
        feasibility_scorer=symbolic_feasibility,
    ),
    "vlm_plus_reachability": SayCanPlanner(
        utility_scorer=vlm_utility,
        feasibility_scorer=reachability_feasibility,
    ),
    # Retained as the ablation that retired itself, not as a recommendation.
    "vlm_plus_critic_affordance": SayCanPlanner(
        utility_scorer=vlm_utility,
        feasibility_scorer=calibrated_affordance,
    ),
    # Say held fixed, Can held fixed, prompt changed: the prompt effect alone.
    "vlm_baseline_prompt": SayCanPlanner(
        utility_scorer=vlm_baseline_utility,
        feasibility_scorer=reachability_feasibility,
    ),
    # Order of the available list changed, everything else held fixed.
    "vlm_shuffled_available": SayCanPlanner(
        utility_scorer=vlm_shuffled_utility,
        feasibility_scorer=reachability_feasibility,
    ),
}
if competence_prior is not None:
    prefix_planners["vlm_plus_competence_prior"] = SayCanPlanner(
        utility_scorer=vlm_utility,
        feasibility_scorer=competence_prior,
    )

# The oracle ignores the planner, so its rollout is shared by every condition.
started = time.perf_counter()
prefix_trajectory = collect_prefix_trajectory(
    num_episodes=NUM_PREFIX_EPISODES,
    seed=EVALUATION_SEED,
    skill_horizon=SKILL_HORIZON,
)
print(
    f"oracle trajectory: {sum(map(len, prefix_trajectory.boundaries))} "
    f"boundaries, {prefix_trajectory.oracle_progress_failures} failures, "
    f"seconds={time.perf_counter() - started:.2f}"
)

prefix_results = {}
for name, planner in prefix_planners.items():
    started = time.perf_counter()
    result = evaluate_planner_prefixes(
        planner,
        num_episodes=NUM_PREFIX_EPISODES,
        seed=EVALUATION_SEED,
        skill_horizon=SKILL_HORIZON,
        trajectory=prefix_trajectory,
    )
    prefix_results[name] = result
    print(
        f"{name:27s} accepted={result['accepted_next_skill_accuracy']:.3f} "
        f"strict={result['strict_accepted_next_skill_accuracy']:.3f} "
        f"parse_valid={result['valid_utility_response_rate']:.3f} "
        f"repaired={result['repaired_utility_response_rate']:.3f} "
        f"valid={result['valid_selection_rate']:.3f} "
        f"grounded={result['grounded_selection_rate']:.3f} "
        f"ungrounded_top={result['ungrounded_top_choice_rate']:.3f} "
        f"rescued={result['feasibility_rescue_rate']:.3f} "
        f"reachable_skills={result['mean_feasible_skills']:.2f} "
        f"seconds={time.perf_counter() - started:.2f}"
    )
    if result["invalid_utility_errors"]:
        print(f"  invalid={result['invalid_utility_errors']}")

# Accepted accuracy per boundary, the position-bias readout: the shuffled
# condition differs from vlm_plus_reachability in list order alone.
BOUNDARY_CONDITIONS = (
    "vlm_plus_reachability",
    "vlm_shuffled_available",
    "vlm_baseline_prompt",
)
num_boundaries = len(prefix_trajectory.boundaries)
print(
    f"\n{'condition':27s} "
    + " ".join(f"{'b' + str(boundary):>5s}" for boundary in range(num_boundaries))
)
for name in BOUNDARY_CONDITIONS:
    traces = prefix_results[name]["traces"]
    cells = []
    for boundary in range(num_boundaries):
        matched = [trace.matched for trace in traces if trace.boundary == boundary]
        cells.append(f"{sum(matched)}/{len(matched)}".rjust(5))
    print(f"{name:27s} " + " ".join(cells))

In [ ]:
INSPECT_CONDITION = (
    "vlm_plus_competence_prior"
    if "vlm_plus_competence_prior" in prefix_results
    else "vlm_plus_reachability"
)

for trace in prefix_results[INSPECT_CONDITION]["traces"]:
    selected = trace.selected_skill.value if trace.selected_skill else "none"
    accepted = ",".join(skill.value for skill in trace.accepted_skills)
    reachable = ",".join(skill.value for skill in trace.feasible_skills) or "none"
    ranking = " > ".join(skill.value for skill in trace.utility_ranking)
    print(
        f"episode={trace.episode} boundary={trace.boundary} "
        f"reference={trace.reference_skill.value:20s} selected={selected:20s} "
        f"accepted=[{accepted}] matched={trace.matched} "
        f"parse_valid={trace.utility_valid} repaired={trace.utility_repaired} "
        f"feasibility={trace.selected_feasibility:.2f}"
    )
    print(
        f"  reachable=[{reachable}] "
        f"top_feasible={trace.utility_top_feasible} "
        f"selected_feasible={trace.selected_feasible}"
    )
    if trace.utility_raw_response is not None:
        print(f"  ranking={ranking}")
        # Under the available_first prompt the model must echo the availability
        # list before the schema lets it choose. Comparing that echo against
        # the true reachable set separates two different failures: a wrong
        # echo is perception, a correct echo followed by an unavailable choice
        # is instruction following.
        try:
            claimed = json.loads(trace.utility_raw_response).get("available_now")
        except (json.JSONDecodeError, AttributeError):
            claimed = None
        if claimed is not None:
            truth = sorted(skill.value for skill in trace.feasible_skills)
            print(
                f"  claimed_available={sorted(claimed)} "
                f"matches_truth={sorted(claimed) == truth}"
            )
        print(f"  raw={trace.utility_raw_response}")
    if trace.utility_error is not None:
        print(f"  error={trace.utility_error}")

## Closed-loop planner/executor ablation

Episodes advance in lockstep by decision index, like the prefix ladder: each decision depends only on its own episode's history, so every running episode is planned in one batched generation and its skills execute together in one compiled loop on device. Each episode keeps its own key, so the results equal running the episodes one by one. With 16 episodes the standard error on a success rate near 0.3 is about 0.11, too wide to separate conditions, so `full` runs 64. An invalid planner response ends its episode, and `planner_errors` says which validation failed. The executor is the one loaded for calibration above, run stochastically.

In [ ]:
if RUN_CLOSED_LOOP:
    closed_loop_results = {}
    for name, planner in prefix_planners.items():
        started = time.perf_counter()
        result = evaluate_skill_chain(
            executor,
            planner=planner,
            num_episodes=NUM_CLOSED_LOOP_EPISODES,
            seed=EVALUATION_SEED,
            skill_horizon=SKILL_HORIZON,
        )
        closed_loop_results[name] = result
        print(
            f"{name:27s} success={result['success_rate']:.3f} "
            f"skills={result['mean_completed_skills']:.2f} "
            f"steps={result['mean_environment_steps']:.1f} "
            f"seconds={time.perf_counter() - started:.2f} "
            f"terminations={result['termination_counts']}"
        )
        print(
            f"  final_failed_skills={result['final_failed_skill_counts']} "
            f"outcomes={result['final_failure_outcome_counts']}"
        )
        if result["planner_error_counts"]:
            print(f"  planner_errors={result['planner_error_counts']}")

## Interpretation

- Low accepted-skill accuracy with valid JSON is a planner reasoning or visual-grounding failure.
- Invalid utility responses are reported separately from states with no viable skill.
- `ungrounded_top` quantifies the VLM's grounding error on its own. It must be identical across every condition that shares `vlm_utility`, because no Can term can change what the VLM ranked first; only `grounded` and `accepted` should move. `vlm_baseline_prompt` is the one condition where it is expected to differ, and that difference is the whole prompt result.
- `vlm_plus_symbolic` to `vlm_plus_reachability` measures how much of the fix comes from knowing what is in reach. `vlm_plus_reachability` to `vlm_plus_competence_prior` measures how much comes from knowing what this executor can finish, now read from measured success rates instead of from a value function that was never trained to answer the question.
- `vlm_plus_critic_affordance` is a control, not a candidate. It should track `vlm_plus_reachability` closely; a large gap in either direction means the critic is doing something, which would contradict the AUC above and is worth investigating before trusting either number.
- Watch `termination_counts` for `no_viable_skill` in the closed loop. Both the reachability term and the competence prior grade rather than veto - the prior has a floor precisely so that a measured 3% success rate stays selectable - because walking toward a distant resource must remain available; a hard gate here turned four-episode scripted chains from 1.0 down to 0.5 success.
- Good prefix accuracy but poor closed-loop success points to executor recovery or planner adaptation after failures.
- The available-first prompt took `ungrounded_top` from 0.75 to 0.03, and with it every Can term's `rescued` to zero: the conditions sharing `vlm_utility` now score identically in the prefix ladder. The planner is still the binding constraint, but its errors moved from grounding to sequencing, which is what the shuffled condition and the per-boundary table probe. Demonstrations or parameter-efficient fine-tuning are only worth considering once the zero-shot prompt ladder has stopped moving.
- A large gap between `accepted` and `strict` means the ranking repair is carrying the result. That is acceptable while the choice itself is right, but a schema that cannot repeat a skill would remove the need for it.